# Step 7 — Comparison with established clinical risk factors

**Lymphoma AI Research** (`dlbcl-prognosis`) · locked analysis plan v1.0, section 11 · code version 07-1.1

Research project only. Nothing in this notebook is a diagnostic or treatment tool.

Each stage starts only after the previous one has passed:

1. **Integrity checks (sections 2–3).** Plan lock, plan files, amendments and every input file are compared with `plan_lock_04.json`, `manifest_04b.json`, `manifest_05.json` and `manifest_06.json`. The frozen model's SHA-256 is compared with five records. The patient-level scores of steps 5 and 6 (`test_lp_05.csv`, `external_lp_06.csv`) must match the hashes in their manifests.
2. **Score check (section 6).** The scores are joined to the clinical table. Patient and death counts, and the LP-alone HR and C-index, must reproduce steps 5 and 6 exactly. No expression file is opened and no gene model is fitted in this notebook.
3. **Comparison (sections 7–10).** In the GSE181063 test set and in both external cohorts: Cox models with IPI, COO and IPI + COO, each with and without the LP (per SD). H3 = likelihood-ratio test (1 df) of Cox(IPI + COO) vs Cox(IPI + COO + LP), primary in the test set. Also: adjusted HR per SD, ΔHarrell C with a 1,000-resample bootstrap (both models refitted in each resample), C-indices alone and combined, missing-IPI diagnostics and, in GSE181063, MHG as a separate COO category.
4. The pre-specified conclusion (§12), figures, outputs, `manifest_07.json` and a summary to paste back (sections 11–13).

TR: Bu not defteri kilitli planın 11. bölümünü uygular. Önce plan kilidi, girdi dosyaları, dondurulmuş model ve 5.–6. adımların hasta düzeyindeki skor dosyaları manifestolarla karşılaştırılır. Sonra skorlar klinik tabloyla birleştirilir; LP'nin tek başına HR ve C değerleri 5. ve 6. adımın sonuçlarını birebir vermelidir. Ancak ondan sonra IPI, COO ve IPI + COO modelleri kurulur ve LP eklenir. Bu not defterinde ifade verisi açılmaz, gen modeli kurulmaz.

## 1. Setup
Mounts Google Drive (in Colab), installs scikit-survival and lifelines if they are missing, and sets the paths. Outside Colab the project folder is taken from the environment variable `DLBCL_PROJECT_DIR`.

TR: Drive'ı bağlar, eksikse kütüphaneleri kurar, klasör yollarını ayarlar. Toplam süre Colab'da yaklaşık 5–10 dakika (çoğu bootstrap).

In [ ]:
import importlib, os, subprocess, sys

for pkg, mod in (('scikit-survival', 'sksurv'), ('lifelines', 'lifelines')):
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/dlbcl-prognosis'
except ImportError:
    PROJECT_DIR = os.environ['DLBCL_PROJECT_DIR']

import datetime, hashlib, json, math, platform, re, time, warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy import stats
import sklearn
import sksurv
import lifelines
from lifelines import CoxPHFitter
from lifelines.exceptions import ConvergenceError, ConvergenceWarning
from lifelines.statistics import logrank_test, proportional_hazard_test
from sksurv.metrics import concordance_index_censored

CODE_VERSION = '07-1.1'
RES04, RES05, RES06 = (f'{PROJECT_DIR}/results/{s}' for s in ('04', '05', '06'))
RES, FIG = f'{PROJECT_DIR}/results/07', f'{PROJECT_DIR}/figures/07'
os.makedirs(RES, exist_ok=True)
os.makedirs(FIG, exist_ok=True)
T_START = time.time()
print('Project folder:', PROJECT_DIR)
print(f'Python {platform.python_version()} | numpy {np.__version__} | pandas {pd.__version__} | '
      f'scikit-survival {sksurv.__version__} | lifelines {lifelines.__version__}')

## 2. Verify the plan lock and the input files
The lock file must be exactly the one recorded after 04a; the two plan files must match the hashes stored in it, and `plan_amendments.md` must have no entries. Every input file is hashed and compared with the lock; the same hashes must appear in `manifest_04b.json`, `manifest_05.json` and `manifest_06.json`. `manifest_06.json` itself must be the file recorded in `progress_step6.md` (SHA-256 starting `0899a7594a1b`).

TR: Kilit dosyası, plan dosyaları, değişiklik kaydı ve bütün girdi dosyaları SHA-256 ile kontrol edilir; aynı özetler üç manifestoda da olmalı. `manifest_06.json` dosyasının özeti de progress_step6'daki değerle başlamalı. Bir şey değiştiyse durur.

In [ ]:
EXPECTED_LOCK_SHA256 = '8d87cf887e2db4770ae3273755d4901cd0d5a2a010fdc729159e007aa129bc7d'    # progress_step4a.md
EXPECTED_MANIFEST06_PREFIX = '0899a7594a1b'                                                      # progress_step6.md

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

def _clean(o):
    """JSON-safe copy: NaN/inf -> null, numpy scalars -> Python."""
    if isinstance(o, dict):
        return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, (np.bool_, bool)):
        return bool(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, (np.floating, float)):
        return float(o) if math.isfinite(float(o)) else None
    return o

def write_json(obj, path):
    with open(path, 'w') as f:
        json.dump(_clean(obj), f, indent=2, sort_keys=False, allow_nan=False)

LOCK_PATH = f'{RES04}/plan_lock_04.json'
assert os.path.exists(LOCK_PATH), 'results/04/plan_lock_04.json not found.'
LOCK_SHA = sha256_file(LOCK_PATH)
assert LOCK_SHA == EXPECTED_LOCK_SHA256, (
    f'The lock file is not the one recorded after 04a ({EXPECTED_LOCK_SHA256[:12]} vs {LOCK_SHA[:12]}). Stop.')
with open(LOCK_PATH) as f:
    LOCK = json.load(f)
for kind in ('json', 'md'):
    path = f"{PROJECT_DIR}/{LOCK[f'plan_{kind}_file']}"
    assert sha256_file(path) == LOCK[f'plan_{kind}_sha256'], f'{path} differs from the locked version. Stop.'
with open(f"{PROJECT_DIR}/{LOCK['plan_json_file']}") as f:
    PLAN = json.load(f)
with open(f"{PROJECT_DIR}/{LOCK['plan_md_file']}") as f:
    PLAN_MD = f.read()
assert PLAN['version'] == '1.0'

AMEND_PATH = f"{PROJECT_DIR}/{PLAN['deviations']['file']}"
amend_text = open(AMEND_PATH).read() if os.path.exists(AMEND_PATH) else ''
AMENDMENTS = re.findall(r'^## Amendment \d+.*$', amend_text, flags=re.M)
if AMENDMENTS:
    raise RuntimeError(f'plan_amendments.md has {len(AMENDMENTS)} entries. This notebook implements plan v1.0 '
                       'without amendments; check before running it.')

M6_PATH = f'{RES06}/manifest_06.json'
assert os.path.exists(M6_PATH), 'results/06/manifest_06.json not found; finish step 6 first.'
M6_SHA = sha256_file(M6_PATH)
assert M6_SHA.startswith(EXPECTED_MANIFEST06_PREFIX), (
    f'manifest_06.json ({M6_SHA[:12]}) is not the file recorded in progress_step6.md ({EXPECTED_MANIFEST06_PREFIX}). '
    'Was step 6 re-run? Stop and check.')
with open(f'{RES04}/manifest_04b.json') as f:
    M4 = json.load(f)
with open(f'{RES05}/manifest_05.json') as f:
    M5 = json.load(f)
with open(M6_PATH) as f:
    M6 = json.load(f)
for name, m in (('manifest_04b', M4), ('manifest_05', M5), ('manifest_06', M6)):
    assert m['plan']['lock_file_sha256'] == LOCK_SHA, f'{name} was produced under a different plan lock. Stop.'
    assert m['plan']['plan_json_sha256'] == LOCK['plan_json_sha256'] and m['plan']['plan_md_sha256'] == LOCK['plan_md_sha256']
    assert m['plan']['amendments'] == 0, f'{name} records plan amendments. Stop.'

INPUT_SHA = {}
print('Input files (SHA-256 vs lock, manifest_04b, manifest_05, manifest_06):')
for key, rel in LOCK['input_files'].items():
    path = f'{PROJECT_DIR}/{rel}'
    assert os.path.exists(path), f'missing input file: {rel}'
    h = sha256_file(path)
    refs = [LOCK['input_sha256'][key]] + [m['inputs_verified_against_lock'][key]['sha256'] for m in (M4, M5, M6)]
    status = 'match' if all(h == r for r in refs) else 'CHANGED'
    INPUT_SHA[key] = {'file': rel, 'sha256': h, 'status': status}
    print(f'  {status:8s} {rel}')
CHANGED = [k for k, v in INPUT_SHA.items() if v['status'] != 'match']
assert not CHANGED, f'Input files changed since the plan was locked: {CHANGED}. Stop.'
print(f"Plan v{PLAN['version']} lock verified (lock {LOCK_SHA[:12]}); no amendments; all {len(INPUT_SHA)} input "
      f'files unchanged; manifest_06.json = {M6_SHA[:12]}')

## 3. Verify the frozen model and the outputs of steps 5 and 6
The SHA-256 of `model_04.json` must equal the value in `progress_step4b.md`, `manifest_04b.json` (two entries), `manifest_05.json` and `manifest_06.json`. Steps 5 and 6 must be complete (`first_results_05.json`, `first_results_06.json`). The two score files used here must match the hashes their own manifests recorded when they were written.

TR: Dondurulmuş modelin özeti beş kayıtla karşılaştırılır. `test_lp_05.csv` ve `external_lp_06.csv` dosyalarının özetleri, yazıldıkları adımın manifestosundaki değerle aynı olmalı; yani skorlar 5. ve 6. adımdan beri değişmemiş olmalı.

In [ ]:
EXPECTED_MODEL_SHA256 = '661e2b80be774252a7fd613116ecc06673a8617ed8a2230bdae56170e3908403'   # progress_step4b.md
MODEL_PATH = f'{RES04}/model_04.json'
MODEL_SHA = sha256_file(MODEL_PATH)
HASH_REFS = {'progress_step4b.md': EXPECTED_MODEL_SHA256,
             'manifest_04b.json, frozen_model.sha256': M4['frozen_model']['sha256'],
             'manifest_04b.json, outputs entry': M4['outputs']['results/04/model_04.json'],
             'manifest_05.json, frozen_model.sha256': M5['frozen_model']['sha256'],
             'manifest_06.json, frozen_model.sha256': M6['frozen_model']['sha256']}
for ref, h in HASH_REFS.items():
    assert MODEL_SHA == h, f'model_04.json ({MODEL_SHA[:12]}) does not match {ref} ({h[:12]}). Stop.'
with open(MODEL_PATH) as f:
    MODEL = json.load(f)
N_GENES = len(MODEL['genes'])
assert N_GENES == M4['frozen_model']['n_genes'] == M6['frozen_model']['n_genes'] == 44

for step, res in (('05', RES05), ('06', RES06)):
    assert os.path.exists(f'{res}/first_results_{step}.json'), f'first_results_{step}.json missing; finish step {step} first.'
assert M5.get('conclusion') and M6.get('conclusion'), 'Step 5 or step 6 has no recorded conclusion.'

SCORE_FILES = {'test': ('results/05/test_lp_05.csv', M5), 'external': ('results/06/external_lp_06.csv', M6)}
SCORE_SHA = {}
for key, (rel, m) in SCORE_FILES.items():
    path = f'{PROJECT_DIR}/{rel}'
    assert os.path.exists(path), f'{rel} not found.'
    h = sha256_file(path)
    assert rel in m['outputs'], f'{rel} is not listed in the outputs of its manifest. Stop.'
    assert h == m['outputs'][rel], f'{rel} ({h[:12]}) differs from its manifest ({m["outputs"][rel][:12]}). Stop.'
    SCORE_SHA[rel] = h
    print(f'  match    {rel} ({h[:12]})')

FIRST_PATH = f'{RES}/first_results_07.json'
print(f'Frozen model verified: SHA-256 {MODEL_SHA[:12]}… = {len(HASH_REFS)} records; {N_GENES} genes')
print(f"Step 5: {M5['conclusion']}")
print(f"Step 6: {M6['conclusion']}")
print('Step 7 run before: ' + ('yes -> results will be compared with first_results_07.json'
                               if os.path.exists(FIRST_PATH) else 'no (first complete run)'))

## 4. Settings from the locked plan
Settings come from `analysis_plan_04.json`. The step-7 entries of the plan are printed in full and their key parts are checked, so the code cannot drift from the plan. Section 11 of the readable plan is printed below for comparison.

TR: Ayarlar kilitli plandan okunur. Plandaki 7. adım kayıtları yazdırılır ve önemli kısımları kontrol edilir; okunur planın 11. bölümü de altta görünür. Not defterinin yaptığıyla karşılaştırabilirsin.

In [ ]:
S7, EP, Q, CR = PLAN['step7_clinical_comparison'], PLAN['endpoints'], PLAN['qc'], PLAN['conclusion_rules']
TIME, EVENT = EP['primary']['time'], EP['primary']['event']
N_BOOT = int(S7['bootstrap_resamples'])
DC_SEED = int(PLAN['seeds']['delta_c_bootstrap'])
FLAG_S = Q['sensitivity_exclusion_flag']
ALPHA = 0.05

assert N_BOOT == 1000
assert DC_SEED == PLAN['master_seed'] + 500 == 20261515
h3 = S7['H3'].lower()
assert 'likelihood-ratio' in h3 and '1 df' in h3 and 'ipi + coo + lp' in h3 and 'test set' in h3 and '0.05' in h3
assert 'complete-case' in S7['missing_ipi'].lower()
assert 'mhg' in S7['coo_sensitivity'].lower() and 'separate' in S7['coo_sensitivity'].lower()
assert any('refit' in s.lower() for s in S7['also_reported'])
rule = CR['adds_information_beyond_ipi_and_coo'].lower()
assert 'h3' in rule and 'test set' in rule and 'both external cohorts' in rule
TRAIN_REF = S7.get('training_reference_from_step3', {})

print('Plan, step 7:')
for k, v in S7.items():
    print(f'  {k}: {v}')
print(f"Conclusion rule: {CR['adds_information_beyond_ipi_and_coo']}")
print(f'Bootstrap: {N_BOOT} resamples, seed {DC_SEED}; outcome {TIME} / {EVENT}')

m = re.search(r'^## 11\..*?(?=^## 12\.)', PLAN_MD, flags=re.M | re.S)
print('\n----- analysis_plan_04.md, section 11 -----\n' + (m.group(0).strip() if m else '(section 11 heading not found)'))

## 5. Functions
Cox models are fitted with lifelines (`CoxPHFitter`, Efron ties, no penalty), as in steps 4b–6. Harrell's C is computed with scikit-survival on each model's linear predictor. COO coding (plan §11): GCB reference; ABC; *other* = UNC, MHG or unclassified. In the sensitivity coding, MHG is a separate category. A covariate column with no variation in a dataset (for example, no "other" patients) is dropped from every model of that dataset, and this is reported.

TR: Cox modelleri lifelines ile kuruluyor (Efron bağları, ceza yok); C-indeksi her modelin doğrusal öngörücüsünden. COO kodlaması: GCB referans, ABC, diğer (UNC, MHG, sınıflanamayan). Duyarlılık kodlamasında MHG ayrı kategori.

In [ ]:
MISSING_TEXT = {'', 'NA', 'NAN', 'NONE', 'MISSING', 'NULL'}

def harrell_c(time_, event, score):
    return float(concordance_index_censored(np.asarray(event).astype(bool), np.asarray(time_, dtype=float),
                                            np.asarray(score, dtype=float))[0])

def norm_coo(s):
    out = s.astype(object).where(s.notna(), None)
    out = out.map(lambda v: None if v is None or str(v).strip().upper() in MISSING_TEXT else str(v).strip().upper())
    return out

def coo_columns(coo, scheme):
    """Dummy columns. '3cat': ABC, other (vs GCB). '4cat': ABC, MHG, other (vs GCB)."""
    d = {'coo_ABC': (coo == 'ABC').astype(float)}
    if scheme == '4cat':
        d['coo_MHG'] = (coo == 'MHG').astype(float)
        d['coo_other'] = (~coo.isin(['GCB', 'ABC', 'MHG'])).astype(float)
    else:
        d['coo_other'] = (~coo.isin(['GCB', 'ABC'])).astype(float)
    return pd.DataFrame(d, index=coo.index)

def fit_cox(df, cols):
    """Unpenalized Cox (Efron). Returns log-likelihood, coefficients, summary, linear predictor and Harrell C."""
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        cph = CoxPHFitter().fit(df[cols + ['T', 'E']], 'T', 'E')
    beta = cph.params_[cols].to_numpy(dtype=float)
    lp = df[cols].to_numpy(dtype=float) @ beta
    return {'cph': cph, 'll': float(cph.log_likelihood_), 'beta': dict(zip(cols, beta)), 'lp': lp,
            'c': harrell_c(df['T'], df['E'], lp),
            'n_conv_warnings': int(sum(issubclass(w.category, ConvergenceWarning) for w in caught))}

def coef_row(fit, col):
    s = fit['cph'].summary.loc[col]
    return {'hr': float(s['exp(coef)']), 'hr_ci_95': [float(s['exp(coef) lower 95%']), float(s['exp(coef) upper 95%'])],
            'coef': float(s['coef']), 'se': float(s['se(coef)']), 'p_wald': float(s['p'])}

def lrt(red, full, df_diff):
    stat = max(0.0, 2.0 * (full['ll'] - red['ll']))
    return {'chi2': float(stat), 'df': int(df_diff), 'p': float(stats.chi2.sf(stat, df_diff))}

def build_models(coo_cols):
    return {'IPI': ['ipi'], 'COO': list(coo_cols), 'LP': ['lp_per_sd'], 'IPI + COO': ['ipi'] + list(coo_cols),
            'IPI + LP': ['ipi', 'lp_per_sd'], 'COO + LP': list(coo_cols) + ['lp_per_sd'],
            'IPI + COO + LP': ['ipi'] + list(coo_cols) + ['lp_per_sd']}

LRT_PAIRS = [('IPI', 'IPI + LP'), ('COO', 'COO + LP'), ('IPI + COO', 'IPI + COO + LP')]
FIT_ERRORS = (ConvergenceError, np.linalg.LinAlgError, ValueError, ZeroDivisionError)

def boot_delta_c(df, red_cols, full_cols, seed, n_boot):
    """Bootstrap of delta Harrell C: both models refitted in every resample (plan §11).
    Index vectors drawn at once with numpy default_rng(seed). Columns with no variation in a resample
    are dropped from both models of that resample (counted)."""
    n = len(df)
    idx = np.random.default_rng(seed).integers(0, n, size=(n_boot, n))
    c_red, c_full, n_dropped, n_failed, n_conv = [], [], 0, 0, 0
    for b in range(n_boot):
        s = df.iloc[idx[b]].reset_index(drop=True)
        keep = [c for c in full_cols if s[c].std() > 0]
        r, f = [c for c in red_cols if c in keep], [c for c in full_cols if c in keep]
        n_dropped += int(len(f) < len(full_cols))
        try:
            fr, ff = fit_cox(s, r), fit_cox(s, f)
        except FIT_ERRORS:
            n_failed += 1
            continue
        n_conv += int(fr['n_conv_warnings'] > 0 or ff['n_conv_warnings'] > 0)
        c_red.append(fr['c']); c_full.append(ff['c'])
    c_red, c_full = np.array(c_red), np.array(c_full)
    d = c_full - c_red
    return {'resamples': n_boot, 'valid': int(len(d)), 'failed': n_failed, 'resamples_with_dropped_column': n_dropped,
            'resamples_with_convergence_warning': n_conv,
            'delta_c_ci_95': [float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))],
            'delta_c_bootstrap_mean': float(d.mean()), 'share_delta_c_le_0': float(np.mean(d <= 0)),
            'c_reduced_ci_95': [float(np.percentile(c_red, 2.5)), float(np.percentile(c_red, 97.5))],
            'c_full_ci_95': [float(np.percentile(c_full, 2.5)), float(np.percentile(c_full, 97.5))]}, d

def fmt_p(p):
    if p is None or not np.isfinite(p):
        return 'n/a'
    return f'{p:.1e}' if p < 0.001 else f'{p:.3f}'

def fmt_hr(r):
    return 'n/a' if r is None else f"{r['hr']:.2f} ({r['hr_ci_95'][0]:.2f}–{r['hr_ci_95'][1]:.2f}), p = {fmt_p(r['p_wald'])}"

def fmt_ci(est, ci, signed=False):
    f = '{:+.3f}' if signed else '{:.3f}'
    return f.format(est) + f" ({f.format(ci[0])} to {f.format(ci[1])})"

def as_bool(s):
    if s.dtype == bool:
        return s
    m = {'true': True, 'false': False, '1': True, '0': False, '1.0': True, '0.0': False}
    out = s.astype(str).str.strip().str.lower().map(m)
    assert out.notna().all(), f'non-boolean values in {s.name}'
    return out.astype(bool)

print('functions defined')

## 6. Load the scores and join the clinical variables (checks)
The test-set and external scores are joined to IPI and COO from `clinical_clean_all.csv`; the training rows are dropped right after loading. Checks, each of which stops the notebook if it fails:

- every test patient is in the test half of `split_GSE181063.csv`, and every external patient is in the right cohort;
- patient and death counts equal the locked populations of steps 5 and 6 (162/74, 229/58, 116/41);
- `lp_per_sd` is the LP standardized in its own analysis population (mean 0, SD 1, perfectly correlated with LP);
- the LP-alone HR per SD and Harrell C reproduce the recorded step-5 and step-6 results to numerical precision.

TR: Skorlar klinik tabloyla birleştirilir. Hasta/ölüm sayıları, `lp_per_sd`'nin doğru ölçeklendiği ve LP'nin tek başına HR ve C değerlerinin 5.–6. adımın kayıtlı sonuçlarını verdiği kontrol edilir. Bunlardan biri tutmazsa birleştirme yanlıştır ve not defteri durur.

In [ ]:
COHORTS = {'GSE181063_test': {'label': 'GSE181063 test', 'role': 'internal test (primary for H3)', 'expected': (162, 74)},
           'GSE10846': {'label': 'GSE10846', 'role': 'external, main', 'expected': (229, 58)},
           'GSE87371': {'label': 'GSE87371', 'role': 'external, supportive', 'expected': (116, 41)}}

clin_all = pd.read_csv(f"{PROJECT_DIR}/{LOCK['input_files']['clinical_all']}")
split = pd.read_csv(f"{PROJECT_DIR}/{LOCK['input_files']['split']}")
test_lp = pd.read_csv(f'{PROJECT_DIR}/results/05/test_lp_05.csv')
ext_lp = pd.read_csv(f'{PROJECT_DIR}/results/06/external_lp_06.csv')
test_lp['cohort'] = 'GSE181063_test'

test_ids = set(split.loc[split['set'] == 'test', 'sample_id'])
assert set(test_lp['sample_id']) <= test_ids, 'test_lp_05.csv contains samples outside the test half of the split. Stop.'
assert set(ext_lp['cohort']) == {'GSE10846', 'GSE87371'}
scores = pd.concat([test_lp[['cohort', 'sample_id', 'lp', 'lp_per_sd', FLAG_S]],
                    ext_lp[['cohort', 'sample_id', 'lp', 'lp_per_sd', FLAG_S]]], ignore_index=True)
assert scores['sample_id'].is_unique
CLIN_COLS = ['sample_id', 'cohort', TIME, EVENT, 'ipi', 'coo', 'age', 'sex']
clin = clin_all.loc[clin_all['sample_id'].isin(scores['sample_id']), CLIN_COLS].rename(columns={'cohort': 'clin_cohort'})
del clin_all                                    # training rows are not used in this step
DATA = scores.merge(clin, on='sample_id', how='left', validate='one_to_one')
assert DATA['clin_cohort'].notna().all(), 'some scored samples have no clinical row. Stop.'
ok_cohort = np.where(DATA['cohort'] == 'GSE181063_test', DATA['clin_cohort'] == 'GSE181063', DATA['clin_cohort'] == DATA['cohort'])
assert ok_cohort.all(), 'cohort labels of scores and clinical table disagree. Stop.'
DATA['T'] = DATA[TIME].astype(float)
DATA['E'] = DATA[EVENT].astype(int)
DATA[FLAG_S] = as_bool(DATA[FLAG_S])
DATA['coo_norm'] = norm_coo(DATA['coo'])
DATA['ipi'] = pd.to_numeric(DATA['ipi'], errors='coerce')
assert DATA['ipi'].dropna().isin([0, 1, 2, 3, 4, 5]).all(), 'IPI values outside 0-5. Stop.'
assert (DATA['T'] > 0).all() and DATA['E'].isin([0, 1]).all()

def rec_count(rec, keys):
    """Patient/death count from a manifest entry, whichever key name that step used."""
    for src in (rec, rec.get('population', {}) if isinstance(rec.get('population'), dict) else {}):
        for k in keys:
            if isinstance(src.get(k), (int, float)) and not isinstance(src.get(k), bool):
                return int(src[k])
    return None

RECORDED = {'GSE181063_test': M5['internal_validation']}
RECORDED.update({g: M6['external_validation'][g] for g in ('GSE10846', 'GSE87371')})
POP_M6 = M6['populations']
REPRO = {}
for coh, c in COHORTS.items():
    d = DATA.loc[DATA['cohort'] == coh]
    n, deaths = len(d), int(d['E'].sum())
    rec = RECORDED[coh]
    assert (n, deaths) == c['expected'], (
        f"{coh}: {n} patients / {deaths} deaths; expected {c['expected']} (progress notes). Stop.")
    rec_n = rec_count(rec, ('n', 'patients', 'n_patients'))
    rec_d = rec_count(rec, ('deaths', 'n_deaths', 'events'))
    if rec_n is not None and rec_d is not None:
        assert (rec_n, rec_d) == (n, deaths), f'{coh}: manifest records {rec_n} / {rec_d}. Stop.'
    else:
        print(f'  note: {coh}: the manifest entry has no patient/death count; checked against the progress notes only.')
    if coh != 'GSE181063_test':
        assert (POP_M6[coh]['patients'], POP_M6[coh]['deaths']) == (n, deaths)
    z = d['lp_per_sd'].to_numpy(float)
    assert abs(z.mean()) < 1e-8 and abs(z.std(ddof=1) - 1) < 1e-8, f'{coh}: lp_per_sd is not standardized. Stop.'
    assert np.corrcoef(z, d['lp'])[0, 1] > 1 - 1e-10, f'{coh}: lp_per_sd is not a positive rescaling of lp. Stop.'
    f = fit_cox(d, ['lp_per_sd'])
    hr, c_all = coef_row(f, 'lp_per_sd')['hr'], harrell_c(d['T'], d['E'], d['lp'])
    rec_hr = rec['hr_per_sd']['hr'] if isinstance(rec['hr_per_sd'], dict) else rec['hr_per_sd']
    assert np.isclose(hr, rec_hr, rtol=1e-6) and np.isclose(c_all, rec['harrell_c'], rtol=1e-9, atol=1e-12), (
        f"{coh}: LP-alone HR {hr:.6f} / C {c_all:.6f} do not reproduce the recorded {rec_hr:.6f} / "
        f"{rec['harrell_c']:.6f}. Stop.")
    REPRO[coh] = {'n': n, 'deaths': deaths, 'hr_per_sd': hr, 'harrell_c': c_all}
    print(f"{c['label']:15s} n {n:3d}, deaths {deaths:3d} | LP alone: HR/SD {hr:.3f}, C {c_all:.3f} "
          f"= step {'5' if coh == 'GSE181063_test' else '6'} | COO values: {d['coo_norm'].value_counts(dropna=False).to_dict()}")
print('All scores and populations reproduce steps 5 and 6.')

## 7. Missing IPI / COO: diagnostics and the complete-case subsets
The plan uses complete-case analysis (patients with both IPI and COO known), so that all models of a cohort are fitted to the same patients and can be compared. For each cohort, patients with and without complete clinical data are compared: deaths, OS (log-rank), the gene score (mean, Mann–Whitney), the LP-alone HR in each group, COO and age. These comparisons are descriptive: they show whether the complete-case subset is selective, not whether the model works.

TR: Eksik IPI (veya COO) olan hastalar çıkarılır; bütün modeller aynı hastalarda kurulur. Eksik ve tam hastalar ölüm oranı, sağkalım, gen skoru, COO ve yaş açısından karşılaştırılır. Bu, eksiksiz vaka alt kümesinin seçici olup olmadığını gösterir (betimleyici).

In [ ]:
MISSING, CC = {}, {}
for coh, c in COHORTS.items():
    d = DATA.loc[DATA['cohort'] == coh].reset_index(drop=True)
    miss_ipi, miss_coo = d['ipi'].isna().to_numpy(), d['coo_norm'].isna().to_numpy()
    cc = ~(miss_ipi | miss_coo)
    r = {'n': len(d), 'deaths': int(d['E'].sum()), 'missing_ipi': int(miss_ipi.sum()), 'missing_coo': int(miss_coo.sum()),
         'deaths_missing_ipi': int(d.loc[miss_ipi, 'E'].sum()), 'complete_case_n': int(cc.sum()),
         'complete_case_deaths': int(d.loc[cc, 'E'].sum())}
    if (~cc).sum() > 0:
        m, o = d.loc[~cc], d.loc[cc]
        lr = logrank_test(o['T'], m['T'], o['E'], m['E'])
        r['incomplete_vs_complete'] = {
            'death_rate_incomplete': float(m['E'].mean()), 'death_rate_complete': float(o['E'].mean()),
            'logrank_p_os': float(lr.p_value),
            'lp_per_sd_mean_incomplete': float(m['lp_per_sd'].mean()), 'lp_per_sd_mean_complete': float(o['lp_per_sd'].mean()),
            'lp_mannwhitney_p': float(stats.mannwhitneyu(m['lp_per_sd'], o['lp_per_sd']).pvalue),
            'age_median_incomplete': float(m['age'].median()), 'age_median_complete': float(o['age'].median()),
            'coo_incomplete': m['coo_norm'].value_counts(dropna=False).rename(index=str).to_dict(),
            'coo_complete': o['coo_norm'].value_counts().to_dict(),
            'lp_hr_per_sd_incomplete': coef_row(fit_cox(m, ['lp_per_sd']), 'lp_per_sd') if m['E'].sum() >= 5 else None,
            'lp_hr_per_sd_complete': coef_row(fit_cox(o, ['lp_per_sd']), 'lp_per_sd')}
    MISSING[coh] = r
    CC[coh] = d.loc[cc].reset_index(drop=True)

    print(f"\n=== {c['label']}: {r['n']} patients / {r['deaths']} deaths; IPI missing {r['missing_ipi']} "
          f"({r['deaths_missing_ipi']} deaths), COO missing {r['missing_coo']} -> complete case "
          f"{r['complete_case_n']} / {r['complete_case_deaths']} deaths")
    if 'incomplete_vs_complete' in r:
        x = r['incomplete_vs_complete']
        print(f"  death rate incomplete {x['death_rate_incomplete']:.1%} vs complete {x['death_rate_complete']:.1%}; "
              f"OS log-rank p = {fmt_p(x['logrank_p_os'])}")
        print(f"  gene score (per SD) mean incomplete {x['lp_per_sd_mean_incomplete']:+.2f} vs complete "
              f"{x['lp_per_sd_mean_complete']:+.2f}, Mann-Whitney p = {fmt_p(x['lp_mannwhitney_p'])}")
        print(f"  LP-alone HR/SD: incomplete {fmt_hr(x['lp_hr_per_sd_incomplete'])}; complete {fmt_hr(x['lp_hr_per_sd_complete'])}")
        print(f"  age median {x['age_median_incomplete']:.0f} vs {x['age_median_complete']:.0f}; "
              f"COO incomplete {x['coo_incomplete']} | complete {x['coo_complete']}")
    else:
        print('  no missing IPI or COO')

pd.DataFrame([{'cohort': k, **{kk: vv for kk, vv in v.items() if kk != 'incomplete_vs_complete'},
               **{f'inc_{kk}': (vv if not isinstance(vv, (dict, list)) else json.dumps(_clean(vv)))
                  for kk, vv in v.get('incomplete_vs_complete', {}).items()}} for k, v in MISSING.items()]
             ).to_csv(f'{RES}/missingness_07.csv', index=False)

## 8. Overlap between the gene score and the clinical factors (descriptive)
How strongly the gene score already tracks IPI and COO in each complete-case subset: Spearman correlation with IPI and the mean score per COO group. A score that mostly re-measures COO or IPI can validate well on its own and still add little; this table helps to read section 9.

TR: Gen skorunun IPI ve COO ile ne kadar örtüştüğü (betimleyici). Skor büyük ölçüde COO'yu ölçüyorsa, tek başına iyi doğrulanır ama ek bilgi az katar.

In [ ]:
OVERLAP = {}
for coh, c in COHORTS.items():
    d = CC[coh]
    rho = stats.spearmanr(d['lp_per_sd'], d['ipi'])
    by_coo = d.groupby('coo_norm')['lp_per_sd'].agg(['size', 'mean']).round(3)
    OVERLAP[coh] = {'spearman_lp_ipi': float(rho.statistic), 'spearman_p': float(rho.pvalue),
                    'lp_per_sd_by_coo': {k: {'n': int(v['size']), 'mean': float(v['mean'])} for k, v in by_coo.iterrows()},
                    'ipi_distribution': d['ipi'].astype(int).value_counts().sort_index().to_dict()}
    print(f"{c['label']:15s} Spearman(LP, IPI) = {rho.statistic:+.2f} (p = {fmt_p(rho.pvalue)}) | mean LP per SD by COO: "
          + ', '.join(f"{k} {v['mean']:+.2f} (n {int(v['size'])})" for k, v in by_coo.iterrows()))

## 9. Primary analysis: H3, adjusted HR and C-indices
In each complete-case subset: seven Cox models (IPI; COO; LP; IPI + COO; IPI + LP; COO + LP; IPI + COO + LP). Likelihood-ratio tests (1 df) for adding the LP to IPI, to COO and to IPI + COO. **H3** is the last of these in the test set (p < 0.05). The adjusted HR is the HR per SD of the LP in Cox(IPI + COO + LP). The C-indices are apparent (same patients used for fitting and evaluation); the ΔC bootstrap in section 10 gives their uncertainty.

TR: Asıl analiz. Her kohortun eksiksiz vaka alt kümesinde yedi Cox modeli kurulur; LP'nin IPI'ya, COO'ya ve IPI + COO'ya eklenmesi olabilirlik oranı testiyle (1 sd) sınanır. H3: test setinde Cox(IPI + COO) ile Cox(IPI + COO + LP) karşılaştırması, p < 0,05. Düzeltilmiş HR, tam modeldeki LP'nin SD başına HR'si.

In [ ]:
def analyse(d, scheme):
    coo = coo_columns(d['coo_norm'], scheme)
    X = pd.concat([d[['T', 'E', 'ipi', 'lp_per_sd']].reset_index(drop=True), coo.reset_index(drop=True)], axis=1)
    coo_cols = [col for col in coo.columns if X[col].std() > 0]
    dropped = [col for col in coo.columns if col not in coo_cols]
    models = build_models(coo_cols)
    fits = {name: fit_cox(X, cols) for name, cols in models.items()}
    out = {'n': len(X), 'deaths': int(X['E'].sum()), 'coo_scheme': scheme, 'coo_columns': coo_cols,
           'dropped_constant_columns': dropped,
           'coo_counts': d['coo_norm'].value_counts().to_dict(),
           'c_index': {name: f['c'] for name, f in fits.items()},
           'log_likelihood': {name: f['ll'] for name, f in fits.items()},
           'convergence_warnings': {name: f['n_conv_warnings'] for name, f in fits.items() if f['n_conv_warnings']},
           'lrt_add_lp': {f'{a} vs {b}': lrt(fits[a], fits[b], len(models[b]) - len(models[a])) for a, b in LRT_PAIRS},
           'hr_per_sd_lp': {name: coef_row(fits[name], 'lp_per_sd') for name in ('LP', 'IPI + LP', 'COO + LP', 'IPI + COO + LP')},
           'full_model_coefficients': {col: coef_row(fits['IPI + COO + LP'], col) for col in models['IPI + COO + LP']},
           'reduced_model_coefficients': {col: coef_row(fits['IPI + COO'], col) for col in models['IPI + COO']}}
    out['delta_c'] = out['c_index']['IPI + COO + LP'] - out['c_index']['IPI + COO']
    try:
        ph = proportional_hazard_test(fits['IPI + COO + LP']['cph'], X[models['IPI + COO + LP'] + ['T', 'E']],
                                      time_transform='rank')
        out['ph_test_full_model_rank'] = {str(i): float(p) for i, p in ph.summary['p'].items()}
    except Exception as e:                  # descriptive only
        out['ph_test_full_model_rank'] = f'not computed: {type(e).__name__}'
    return out, X, models

PRIMARY, XMAT, MODELS = {}, {}, {}
for coh, c in COHORTS.items():
    R, X, mods = analyse(CC[coh], '3cat')
    PRIMARY[coh], XMAT[coh], MODELS[coh] = R, X, mods
    H = R['lrt_add_lp']['IPI + COO vs IPI + COO + LP']
    print(f"\n=== {c['label']} ({c['role']}) — complete case n = {R['n']}, deaths = {R['deaths']}; COO {R['coo_counts']}"
          + (f"; dropped (no variation): {R['dropped_constant_columns']}" if R['dropped_constant_columns'] else ''))
    print('  Harrell C: ' + ' | '.join(f'{k} {v:.3f}' for k, v in R['c_index'].items()))
    for pair, t in R['lrt_add_lp'].items():
        tag = '  <- H3' if (coh == 'GSE181063_test' and pair.startswith('IPI + COO vs')) else ''
        print(f"  LRT {pair:28s} chi2 = {t['chi2']:6.2f} (df {t['df']}), p = {fmt_p(t['p'])}{tag}")
    for name, h in R['hr_per_sd_lp'].items():
        print(f"  HR per SD of LP, {'unadjusted' if name == 'LP' else 'adjusted for ' + name[:-5]:28s}: {fmt_hr(h)}")
    print('  Full model: ' + '; '.join(f"{k} HR {v['hr']:.2f} ({v['hr_ci_95'][0]:.2f}–{v['hr_ci_95'][1]:.2f})"
                                       for k, v in R['full_model_coefficients'].items()))
    print(f"  delta C (IPI + COO + LP minus IPI + COO) = {R['delta_c']:+.3f}")
    if isinstance(R['ph_test_full_model_rank'], dict):
        print('  PH check, full model (Schoenfeld, rank; descriptive): '
              + ', '.join(f'{k} p = {fmt_p(v)}' for k, v in R['ph_test_full_model_rank'].items()))
    if R['convergence_warnings']:
        print('  convergence warnings:', R['convergence_warnings'])

H3 = PRIMARY['GSE181063_test']['lrt_add_lp']['IPI + COO vs IPI + COO + LP']
H3_MET = bool(H3['p'] < ALPHA)
ADJ_HR = {coh: PRIMARY[coh]['hr_per_sd_lp']['IPI + COO + LP'] for coh in COHORTS}
print(f"\nH3 (test set): LRT chi2 = {H3['chi2']:.2f}, p = {fmt_p(H3['p'])} -> {'MET' if H3_MET else 'NOT MET'}")
if TRAIN_REF:
    print(f"Training reference (step 3, n {TRAIN_REF.get('n')}): COO alone C {TRAIN_REF.get('coo_alone_c')}, "
          f"IPI + COO C {TRAIN_REF.get('ipi_plus_coo_c')}")

## 10. ΔHarrell C with bootstrap CI (plan §11)
1,000 resamples of the complete-case patients with replacement (seed 20261515, all index vectors drawn at once, separately for each cohort). In every resample Cox(IPI + COO) and Cox(IPI + COO + LP) are **both refitted**, and ΔC = C(full) − C(reduced) is computed on the resample. Percentile 95% interval. Takes a few minutes.

TR: 1.000 bootstrap örnekleminde iki model de her seferinde yeniden kurulur ve C farkı hesaplanır; yüzdelik %95 GA. Birkaç dakika sürer.

In [ ]:
BOOT, BOOT_D = {}, {}
for coh, c in COHORTS.items():
    t0 = time.time()
    m = MODELS[coh]
    BOOT[coh], BOOT_D[coh] = boot_delta_c(XMAT[coh], m['IPI + COO'], m['IPI + COO + LP'], DC_SEED, N_BOOT)
    b = BOOT[coh]
    print(f"{c['label']:15s} delta C = {fmt_ci(PRIMARY[coh]['delta_c'], b['delta_c_ci_95'], signed=True)}; "
          f"{b['valid']}/{b['resamples']} valid resamples, {b['failed']} failed, "
          f"{b['resamples_with_dropped_column']} with a dropped column, {b['resamples_with_convergence_warning']} with "
          f"convergence warnings; share of resamples with delta C <= 0: {b['share_delta_c_le_0']:.2f} ({time.time() - t0:.0f} s)")

## 11. Sensitivity analysis: MHG as a separate COO category (GSE181063 test set)
Same complete-case patients and the same tests, with COO coded GCB (reference), ABC, MHG and other (UNC). The ΔC bootstrap uses the same seed. MHG exists only in GSE181063, so this analysis is done there only.

TR: Test setinde COO dört kategoriyle (GCB referans, ABC, MHG, diğer) kodlanarak H3, düzeltilmiş HR ve ΔC yeniden hesaplanır.

In [ ]:
SENS_MHG, _, mods_mhg = analyse(CC['GSE181063_test'], '4cat')
n_mhg = int((CC['GSE181063_test']['coo_norm'] == 'MHG').sum())
X_mhg = pd.concat([CC['GSE181063_test'][['T', 'E', 'ipi', 'lp_per_sd']].reset_index(drop=True),
                   coo_columns(CC['GSE181063_test']['coo_norm'], '4cat').reset_index(drop=True)], axis=1)
SENS_MHG['n_mhg'] = n_mhg
SENS_MHG['bootstrap'], _ = boot_delta_c(X_mhg, mods_mhg['IPI + COO'], mods_mhg['IPI + COO + LP'], DC_SEED, N_BOOT)
t = SENS_MHG['lrt_add_lp']['IPI + COO vs IPI + COO + LP']
print(f"MHG patients in the complete-case test set: {n_mhg}" + ('' if n_mhg else ' -> MHG column dropped (no variation)'))
print(f"LRT IPI + COO(4) vs IPI + COO(4) + LP: chi2 = {t['chi2']:.2f}, p = {fmt_p(t['p'])}")
print(f"Adjusted HR per SD: {fmt_hr(SENS_MHG['hr_per_sd_lp']['IPI + COO + LP'])}")
print(f"C: IPI + COO(4) {SENS_MHG['c_index']['IPI + COO']:.3f}, + LP {SENS_MHG['c_index']['IPI + COO + LP']:.3f}; "
      f"delta C = {fmt_ci(SENS_MHG['delta_c'], SENS_MHG['bootstrap']['delta_c_ci_95'], signed=True)}")
if 'coo_MHG' in SENS_MHG['full_model_coefficients']:
    print(f"MHG vs GCB in the full model: {fmt_hr(SENS_MHG['full_model_coefficients']['coo_MHG'])}")

## 12. Figures
1. Forest plot: HR per SD of the gene score in each complete-case subset, unadjusted and adjusted for IPI, COO and IPI + COO (plus the MHG sensitivity analysis).
2. Harrell C of IPI, COO, LP, IPI + COO and IPI + COO + LP in each complete-case subset.
3. Bootstrap distributions of ΔC.

Saved in `figures/07/`.

TR: Orman grafiği (ham ve düzeltilmiş HR), modellerin C-indeksleri ve ΔC bootstrap dağılımları.

In [ ]:
BLUE, ORANGE, AQUA, INK, MUTED, GRID = '#2a78d6', '#eb6834', '#1baf7a', '#0b0b0b', '#52514e', '#e4e3df'
plt.rcParams.update({'axes.edgecolor': MUTED, 'axes.labelcolor': INK, 'xtick.color': MUTED, 'ytick.color': MUTED,
                     'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 10})
FIG_FILES = []

rows = []
for coh, c in COHORTS.items():
    for name, lab in (('LP', 'unadjusted'), ('IPI + LP', 'adj. IPI'), ('COO + LP', 'adj. COO'), ('IPI + COO + LP', 'adj. IPI + COO')):
        h = PRIMARY[coh]['hr_per_sd_lp'][name]
        rows.append((f"{c['label']} · {lab}", h['hr'], *h['hr_ci_95'], name == 'IPI + COO + LP'))
    if coh == 'GSE181063_test':
        h = SENS_MHG['hr_per_sd_lp']['IPI + COO + LP']
        rows.append((f"{c['label']} · adj. IPI + COO (MHG separate)", h['hr'], *h['hr_ci_95'], False))
fig, ax = plt.subplots(figsize=(8.5, 0.42 * len(rows) + 1.3))
for i, (lab, hr, lo, hi, main) in enumerate(rows[::-1]):
    ax.plot([lo, hi], [i, i], color=INK if main else MUTED, lw=2, solid_capstyle='round')
    ax.plot(hr, i, 'o', ms=8 if main else 5.5, color=ORANGE if main else MUTED)
    ax.text(hi * 1.04, i, f'{hr:.2f} ({lo:.2f}–{hi:.2f})', va='center', color=MUTED, fontsize=8.5)
ax.axvline(1, color=MUTED, lw=1, ls='--')
ax.set_xscale('log')
lo_all, hi_all = min(r[2] for r in rows), max(r[3] for r in rows)
ticks = [t for t in (0.5, 0.75, 1, 1.5, 2, 3, 4) if lo_all / 1.3 <= t <= hi_all * 1.3]
ax.set_xticks(ticks); ax.set_xticklabels([f'{t:g}' for t in ticks]); ax.minorticks_off()
ax.set_xlim(min(lo_all, 0.9) / 1.1, hi_all * 1.7)
ax.set_yticks(range(len(rows))); ax.set_yticklabels([r[0] for r in rows[::-1]], fontsize=8.5)
ax.set_xlabel('HR per SD of the gene score (95% CI, log scale), complete-case subsets')
ax.grid(axis='x', color=GRID, lw=0.8)
ax.set_title('Gene score: unadjusted and adjusted for IPI and COO', loc='left', color=INK)
fig.tight_layout(); p = f'{FIG}/forest_adjusted_hr_07.png'; fig.savefig(p, dpi=150); plt.show(); plt.close(fig)
FIG_FILES.append(p)

order = ['IPI', 'COO', 'LP', 'IPI + COO', 'IPI + COO + LP']
colors = {'IPI': MUTED, 'COO': MUTED, 'LP': BLUE, 'IPI + COO': INK, 'IPI + COO + LP': ORANGE}
fig, axes = plt.subplots(1, len(COHORTS), figsize=(11, 3.4), sharex=True)
for ax, (coh, c) in zip(axes, COHORTS.items()):
    cs = PRIMARY[coh]['c_index']
    for i, name in enumerate(order[::-1]):
        ax.plot([0.5, cs[name]], [i, i], color=GRID, lw=1.5)
        ax.plot(cs[name], i, 'o', ms=8, color=colors[name])
        ax.text(cs[name] + 0.008, i, f'{cs[name]:.3f}', va='center', fontsize=8.5, color=MUTED)
    ax.set_yticks(range(len(order))); ax.set_yticklabels(order[::-1])
    ax.set_title(f"{c['label']} (n {PRIMARY[coh]['n']})", loc='left', fontsize=10)
    ax.set_xlim(0.5, max(0.85, max(cs.values()) + 0.06)); ax.grid(axis='x', color=GRID, lw=0.8)
axes[1].set_xlabel("Harrell's C (apparent), complete-case subsets")
fig.tight_layout(); p = f'{FIG}/c_index_models_07.png'; fig.savefig(p, dpi=150); plt.show(); plt.close(fig)
FIG_FILES.append(p)

fig, axes = plt.subplots(1, len(COHORTS), figsize=(11, 3.2), sharey=True)
for ax, (coh, c) in zip(axes, COHORTS.items()):
    d = BOOT_D[coh]
    ax.hist(d, bins=40, color=BLUE, alpha=0.8, edgecolor='white', lw=0.4)
    ax.axvline(0, color=INK, lw=1, ls='--')
    ax.axvline(PRIMARY[coh]['delta_c'], color=ORANGE, lw=2)
    lo, hi = BOOT[coh]['delta_c_ci_95']
    ax.set_title(f"{c['label']}: ΔC {PRIMARY[coh]['delta_c']:+.3f} ({lo:+.3f} to {hi:+.3f})", loc='left', fontsize=9.5)
    ax.set_xlabel('ΔC, IPI + COO + LP minus IPI + COO')
axes[0].set_ylabel('bootstrap resamples')
fig.tight_layout(); p = f'{FIG}/delta_c_bootstrap_07.png'; fig.savefig(p, dpi=150); plt.show(); plt.close(fig)
FIG_FILES.append(p)

## 13. Red flags and the pre-specified conclusion
Red flag (plan §13): any C-index > 0.90. Conclusion wording is fixed by plan §12: "Adds information beyond IPI and COO" = H3 met in the test set **and** adjusted HR > 1 (point estimate) in both external cohorts. Otherwise the result is reported as negative or inconclusive, with the same detail.

TR: Kırmızı bayrak kontrolü ve planın 12. bölümündeki önceden sabitlenmiş sonuç cümlesi.

In [ ]:
RED_FLAGS, EXTRA_WARNINGS = [], []
for coh in COHORTS:
    for name, cval in PRIMARY[coh]['c_index'].items():
        if cval > 0.90:
            RED_FLAGS.append(f'{coh} / {name}: C = {cval:.3f} > 0.90')
    if BOOT[coh]['failed'] > 0.05 * N_BOOT:
        EXTRA_WARNINGS.append(f"{coh}: {BOOT[coh]['failed']} bootstrap resamples failed (> 5%)")
if H3_MET and ADJ_HR['GSE181063_test']['hr'] < 1:
    EXTRA_WARNINGS.append('H3 is significant but the adjusted HR in the test set is < 1 (opposite direction)')

EXT_ADJ_POS = {g: bool(ADJ_HR[g]['hr'] > 1) for g in ('GSE10846', 'GSE87371')}
ADDS_INFO = bool(H3_MET and all(EXT_ADJ_POS.values()))
detail = (f"H3 in the test set {'met' if H3_MET else 'not met'} (LRT chi2 = {H3['chi2']:.2f}, p = {fmt_p(H3['p'])}; "
          f"adjusted HR per SD {ADJ_HR['GSE181063_test']['hr']:.2f}); adjusted HR per SD "
          + '; '.join(f"{g} {ADJ_HR[g]['hr']:.2f} ({'> 1' if EXT_ADJ_POS[g] else '<= 1'})" for g in EXT_ADJ_POS))
if ADDS_INFO:
    CONCLUSION = 'Adds information beyond IPI and COO (' + detail + ')'
else:
    CONCLUSION = ('Not shown to add information beyond IPI and COO per plan §12; reported as negative or '
                  'inconclusive (' + detail + ')')
SUPPORT = {coh: {'adjusted HR 95% CI excludes 1': bool(ADJ_HR[coh]['hr_ci_95'][0] > 1),
                 'LRT p < 0.05': bool(PRIMARY[coh]['lrt_add_lp']['IPI + COO vs IPI + COO + LP']['p'] < ALPHA),
                 'delta C 95% CI above 0': bool(BOOT[coh]['delta_c_ci_95'][0] > 0)} for coh in COHORTS}

print('Red flags:', RED_FLAGS if RED_FLAGS else 'none')
print('Extra warnings:', EXTRA_WARNINGS if EXTRA_WARNINGS else 'none')
for coh, s in SUPPORT.items():
    print(f"{COHORTS[coh]['label']:15s} {s}")
print('\nCONCLUSION (plan §12):', CONCLUSION)
if RED_FLAGS:
    print('\nSTOP: a red flag was raised. Look for an error before reporting anything.')

## 14. Save the results, the manifest and the summary
All results go to `results/07/`, figures to `figures/07/`, and `manifest_07.json` records the SHA-256 of every input and output. The key results of the first complete run are written once to `first_results_07.json` and never overwritten; a later run is compared with it (plan §17).

TR: Bütün çıktılar kaydedilir. İlk tam çalıştırmanın ana sonuçları `first_results_07.json` dosyasına bir kez yazılır. En alttaki özeti bana yapıştır.

In [ ]:
CTAB = pd.DataFrame([{'cohort': coh, 'model': name, 'harrell_c': cval, 'n': PRIMARY[coh]['n'], 'deaths': PRIMARY[coh]['deaths']}
                     for coh in COHORTS for name, cval in PRIMARY[coh]['c_index'].items()])
CTAB.to_csv(f'{RES}/c_index_models_07.csv', index=False)
LRT_TAB = pd.DataFrame([{'cohort': coh, 'comparison': pair, **t} for coh in COHORTS
                        for pair, t in PRIMARY[coh]['lrt_add_lp'].items()]
                       + [{'cohort': 'GSE181063_test (MHG separate)', 'comparison': pair, **t}
                          for pair, t in SENS_MHG['lrt_add_lp'].items()])
LRT_TAB.to_csv(f'{RES}/lrt_07.csv', index=False)
HR_TAB = pd.DataFrame([{'cohort': coh, 'model': name, 'hr_per_sd': h['hr'], 'ci_low': h['hr_ci_95'][0], 'ci_high': h['hr_ci_95'][1],
                        'p_wald': h['p_wald']} for coh in COHORTS for name, h in PRIMARY[coh]['hr_per_sd_lp'].items()]
                      + [{'cohort': 'GSE181063_test (MHG separate)', 'model': name, 'hr_per_sd': h['hr'], 'ci_low': h['hr_ci_95'][0],
                          'ci_high': h['hr_ci_95'][1], 'p_wald': h['p_wald']} for name, h in SENS_MHG['hr_per_sd_lp'].items()])
HR_TAB.to_csv(f'{RES}/hr_lp_07.csv', index=False)
pd.DataFrame({coh: pd.Series(BOOT_D[coh]) for coh in COHORTS}).to_csv(f'{RES}/delta_c_bootstrap_07.csv', index=False)
CLIN_RESULTS = {'populations': MISSING, 'reproduction_of_steps_5_6': REPRO, 'overlap_descriptive': OVERLAP,
                'primary': PRIMARY, 'delta_c_bootstrap': BOOT, 'sensitivity_mhg_separate': SENS_MHG,
                'H3': {'definition': S7['H3'], **H3, 'met': H3_MET}, 'adjusted_hr_per_sd': ADJ_HR,
                'supporting_evidence': SUPPORT, 'red_flags': RED_FLAGS, 'extra_warnings': EXTRA_WARNINGS,
                'conclusion': CONCLUSION}
write_json(CLIN_RESULTS, f'{RES}/clinical_comparison_07.json')

KEY = {'code_version': CODE_VERSION, 'model_sha256': MODEL_SHA}
for coh in COHORTS:
    P, B = PRIMARY[coh], BOOT[coh]
    t = P['lrt_add_lp']['IPI + COO vs IPI + COO + LP']
    KEY[coh] = {'complete_case_n': P['n'], 'complete_case_deaths': P['deaths'], 'lrt_chi2': t['chi2'], 'lrt_p': t['p'],
                'adjusted_hr_per_sd': ADJ_HR[coh]['hr'], 'adjusted_hr_ci_95': ADJ_HR[coh]['hr_ci_95'],
                'c_ipi_coo': P['c_index']['IPI + COO'], 'c_ipi_coo_lp': P['c_index']['IPI + COO + LP'],
                'delta_c': P['delta_c'], 'delta_c_ci_95': B['delta_c_ci_95']}
KEY['mhg_separate'] = {'lrt_p': SENS_MHG['lrt_add_lp']['IPI + COO vs IPI + COO + LP']['p'],
                       'adjusted_hr_per_sd': SENS_MHG['hr_per_sd_lp']['IPI + COO + LP']['hr'],
                       'delta_c': SENS_MHG['delta_c'], 'delta_c_ci_95': SENS_MHG['bootstrap']['delta_c_ci_95']}
KEY['h3_met'], KEY['adds_information_beyond_ipi_and_coo'] = H3_MET, ADDS_INFO

def _same(a, b):
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    if isinstance(a, list) and isinstance(b, list):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, float) or isinstance(b, float):
        return a is not None and b is not None and bool(np.isclose(a, b, rtol=1e-9, atol=1e-12))
    return a == b

KEY_CLEAN = _clean(KEY)
RUN_UTC = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')
if not os.path.exists(FIRST_PATH):
    write_json({'note': 'Written once, at the first complete run of step 7; never overwritten (plan §17).',
                'written_utc': RUN_UTC, **KEY_CLEAN}, FIRST_PATH)
    FIRST_STATUS = 'first complete run: first_results_07.json written'
else:
    with open(FIRST_PATH) as f:
        FIRST = json.load(f)
    same = all(_same(KEY_CLEAN[k], FIRST.get(k)) for k in KEY_CLEAN)
    FIRST_STATUS = ('re-run: identical to first_results_07.json' if same else
                    're-run: DIFFERS from first_results_07.json — report both (plan §17) and find out why')
print(FIRST_STATUS)

SUMMARY = [f'Step 7 clinical comparison ({CODE_VERSION}) | model {MODEL_SHA[:12]} | lock {LOCK_SHA[:12]} | {FIRST_STATUS}']
for coh, c in COHORTS.items():
    P, B, Mi = PRIMARY[coh], BOOT[coh], MISSING[coh]
    t = P['lrt_add_lp']['IPI + COO vs IPI + COO + LP']
    SUMMARY.append(f"{c['label']} ({c['role']}): all {Mi['n']}/{Mi['deaths']} d; IPI missing {Mi['missing_ipi']}, "
                   f"COO missing {Mi['missing_coo']} -> complete case {P['n']}/{P['deaths']} d; COO {P['coo_counts']}")
    if 'incomplete_vs_complete' in Mi:
        x = Mi['incomplete_vs_complete']
        SUMMARY.append(f"  missing-data check: death rate {x['death_rate_incomplete']:.1%} vs {x['death_rate_complete']:.1%}, "
                       f"OS log-rank p {fmt_p(x['logrank_p_os'])}; LP mean {x['lp_per_sd_mean_incomplete']:+.2f} vs "
                       f"{x['lp_per_sd_mean_complete']:+.2f} (MW p {fmt_p(x['lp_mannwhitney_p'])})")
    SUMMARY.append('  C: ' + ' | '.join(f'{k} {v:.3f}' for k, v in P['c_index'].items()))
    SUMMARY.append('  LRT +LP: ' + ' | '.join(f"{k.split(' vs ')[0]}: chi2 {v['chi2']:.2f}, p {fmt_p(v['p'])}"
                                              for k, v in P['lrt_add_lp'].items()))
    SUMMARY.append('  HR/SD of LP: ' + ' | '.join(f"{k}: {v['hr']:.2f} ({v['hr_ci_95'][0]:.2f}–{v['hr_ci_95'][1]:.2f})"
                                                  for k, v in P['hr_per_sd_lp'].items()))
    SUMMARY.append(f"  full model: " + '; '.join(f"{k} {v['hr']:.2f} ({v['hr_ci_95'][0]:.2f}–{v['hr_ci_95'][1]:.2f})"
                                                 for k, v in P['full_model_coefficients'].items()))
    SUMMARY.append(f"  delta C {fmt_ci(P['delta_c'], B['delta_c_ci_95'], signed=True)} "
                   f"[{B['valid']}/{B['resamples']} valid, {B['failed']} failed]; "
                   f"Spearman(LP, IPI) {OVERLAP[coh]['spearman_lp_ipi']:+.2f}")
SM = SENS_MHG
SUMMARY.append(f"MHG separate (test, n MHG {SM['n_mhg']}): LRT p {fmt_p(SM['lrt_add_lp']['IPI + COO vs IPI + COO + LP']['p'])}, "
               f"adj HR {fmt_hr(SM['hr_per_sd_lp']['IPI + COO + LP'])}, delta C "
               f"{fmt_ci(SM['delta_c'], SM['bootstrap']['delta_c_ci_95'], signed=True)}")
SUMMARY.append(f"H3 (test): chi2 {H3['chi2']:.2f}, p {fmt_p(H3['p'])} -> {'MET' if H3_MET else 'NOT MET'}")
SUMMARY.append(f"Red flags: {RED_FLAGS or 'none'} | extra warnings: {EXTRA_WARNINGS or 'none'}")
SUMMARY.append(f'CONCLUSION: {CONCLUSION}')
with open(f'{RES}/summary_07.txt', 'w') as f:
    f.write('\n'.join(SUMMARY) + '\n')

OUT_FILES = ['missingness_07.csv', 'c_index_models_07.csv', 'lrt_07.csv', 'hr_lp_07.csv', 'delta_c_bootstrap_07.csv',
             'clinical_comparison_07.json', 'first_results_07.json', 'summary_07.txt']
MANIFEST = {
    'step': '07', 'notebook': '07_clinical_comparison.ipynb', 'code_version': CODE_VERSION, 'created_utc': RUN_UTC,
    'plan': {'version': PLAN['version'], 'plan_json_sha256': LOCK['plan_json_sha256'], 'plan_md_sha256': LOCK['plan_md_sha256'],
             'lock_file_sha256': LOCK_SHA, 'amendments': len(AMENDMENTS), 'sections_implemented': ['11', '12', '13']},
    'frozen_model': {'file': 'results/04/model_04.json', 'sha256': MODEL_SHA, 'matches': list(HASH_REFS), 'n_genes': N_GENES},
    'inputs_verified_against_lock': INPUT_SHA,
    'upstream_outputs_verified': {**{rel: {'sha256': h, 'matches': 'manifest_05.json' if '/05/' in rel else 'manifest_06.json'}
                                     for rel, h in SCORE_SHA.items()},
                                  'results/06/manifest_06.json': {'sha256': M6_SHA, 'matches': 'progress_step6.md (prefix)'}},
    'data_access': {'opened': ['results/05/test_lp_05.csv', 'results/06/external_lp_06.csv',
                               'rows of data/clean/clinical_clean_all.csv for the scored test and external patients (OS, IPI, COO, age, sex)',
                               'data/clean/split_GSE181063.csv (test membership check)'],
                    'not_opened': ['every expression file (training, test, external)'],
                    'fitted': 'unpenalized Cox models of clinical variables and the frozen score; no gene model'},
    'populations': MISSING, 'reproduction_of_steps_5_6': REPRO,
    'clinical_comparison': KEY_CLEAN, 'H3_met': H3_MET, 'supporting_evidence': SUPPORT,
    'conclusion': CONCLUSION, 'red_flags': RED_FLAGS, 'extra_warnings': EXTRA_WARNINGS, 'first_results_status': FIRST_STATUS,
    'implementation_notes': [
        'Order: plan lock, plan files, amendments and inputs (vs lock, manifest_04b, manifest_05, manifest_06); manifest_06 hash prefix vs progress_step6.md; frozen model hash (5 records); test_lp_05.csv and external_lp_06.csv vs the outputs of their manifests; joins checked by reproducing the recorded counts, LP-alone HR per SD and Harrell C.',
        'Score: lp_per_sd from steps 5 and 6 (LP standardized with the mean and SD, ddof 1, of each analysis population: test set 162, GSE10846 229, GSE87371 116).',
        'COO coding: text stripped and upper-cased; empty/NA treated as missing; GCB reference, ABC, other = any other non-missing value (UNC, MHG, unclassified). Sensitivity: GCB, ABC, MHG, other. IPI used as linear 0-5.',
        'Complete case: patients with both IPI and COO known; all seven models of a cohort fitted to the same patients. Missing COO is not covered by the plan text and is handled like missing IPI.',
        'Cox models: lifelines CoxPHFitter (Efron ties, no penalty). LRT = 2 x difference in log partial likelihood, chi-square with df = number of added terms (1). Adjusted HR = HR per SD of lp_per_sd in Cox(IPI + COO + LP), Wald 95% CI.',
        'Harrell C: scikit-survival concordance_index_censored on each model linear predictor (apparent, same patients).',
        'Delta C bootstrap: 1,000 resamples of complete-case patients, numpy default_rng(20261515).integers, all index vectors drawn at once, separately per cohort and for the MHG sensitivity analysis; both models refitted in every resample; C computed on the resample; percentile interval. Columns without variation in a resample dropped from both models of that resample (counted).',
        'A covariate column without variation in a whole complete-case subset is dropped from all models of that subset (reported in dropped_constant_columns).',
        'Missingness diagnostics (descriptive): patients with vs without complete IPI/COO compared on death rate, OS (log-rank), lp_per_sd (Mann-Whitney), LP-alone HR, age and COO.',
        'PH check: lifelines proportional_hazard_test on the full model, rank transform; descriptive only.',
        'Conclusion: plan 12, adjusted HR > 1 judged on the point estimate in both external cohorts.'],
    'environment': {'python': platform.python_version(), 'platform': platform.platform(), 'cpu_count': os.cpu_count(),
                    'numpy': np.__version__, 'pandas': pd.__version__, 'scipy': scipy.__version__,
                    'scikit-learn': sklearn.__version__, 'scikit-survival': sksurv.__version__,
                    'lifelines': lifelines.__version__, 'matplotlib': matplotlib.__version__},
    'runtime_seconds_this_run': round(time.time() - T_START, 1),
    'outputs': {**{f'results/07/{f}': sha256_file(f'{RES}/{f}') for f in OUT_FILES},
                **{os.path.relpath(p, PROJECT_DIR): sha256_file(p) for p in FIG_FILES}},
}
write_json(MANIFEST, f'{RES}/manifest_07.json')
print(f"manifest_07.json written (SHA-256 {sha256_file(f'{RES}/manifest_07.json')[:12]})\n")
print('=' * 30, 'SUMMARY — paste this back', '=' * 30)
print('\n'.join(SUMMARY))

TR: Bitti. Yukarıdaki özeti sohbete yapıştır. Sonra **Çalışma zamanı → Bağlantıyı kes ve çalışma zamanını sil** ile oturumu kapat.